# 5주차. 카나의 자율 약속 잡기

**부제:** MCP 서버 Tool Call 분리하기

## 0. 목표

이번 주에는 MCP가 왜 "tool 제공자"와 "agent 실행 코드"를 분리하는지 payload 모양으로 이해한다. 실제 문제 코드는 별도 repo에서 다루고, 이 노트북은 MCP tool payload와 trace를 읽는 기준을 정리한다.

성취 기준:

- Python 함수 tool과 MCP tool의 차이를 tool 제공 위치 기준으로 설명할 수 있다.
- MCP 서버가 반환해야 하는 payload의 핵심 필드를 구분할 수 있다.
- 단일 agent가 MCP tool을 호출했을 때 확인할 trace 기준을 말할 수 있다.

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

5주차의 실제 MCP server/client 문제 코드는 별도 문제 repo에서 작성한다. 이 레포에서는 노트북으로 tool schema, payload, trace 관찰 기준을 먼저 맞춘다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → MCP payload shape → agent trace 확인 기준 순서로 진행한다.

In [ ]:
import json
from datetime import datetime, timezone

def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))

def calendar_check_availability_payload(members: list[str], date: str) -> dict:
    return {
        "server": "kanamate-calendar",
        "tool": "calendar.check_availability",
        "arguments": {"members": members, "date": date},
        "available_slots": [f"{date} 10:00", f"{date} 15:00"],
        "checked_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

def calendar_create_event_payload(title: str, date: str, start_time: str, members: list[str]) -> dict:
    event_id = f"event-{date}-{start_time}".replace(":", "")
    return {
        "server": "kanamate-calendar",
        "tool": "calendar.create_event",
        "arguments": {"title": title, "date": date, "start_time": start_time, "members": members},
        "event_id": event_id,
        "status": "created",
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

show_json({"mcp_server": "kanamate-calendar", "expected_tools": ["calendar_check_availability", "calendar_create_event"]})

## 2. 개념

오늘의 큰 그림: MCP는 agent 코드 안에 tool 함수를 직접 넣는 대신, 외부 서버가 표준 방식으로 tool 목록과 실행 결과를 제공하게 해준다. 이 노트북에서는 SQLite 저장을 다루지 않고 MCP payload만 본다.

반드시 이해할 것:

- MCP 서버는 tool 제공자다.
- Agent 실행 코드는 MCP 서버에서 tool 목록을 읽는 client다.
- Agent는 MCP 서버에서 읽어온 tool을 호출한다.
- 성공 여부는 MCP payload의 `server`, `tool`, `arguments`, `event_id`, `status`로 확인한다.

지금은 몰라도 되는 것:

- FastMCP 서버 런타임 내부 구조
- `MultiServerMCPClient` 구현 세부
- 원격 배포와 인증

막혔을 때 볼 trace: MCP server URL, MCP tool name, MCP tool result payload.

## 3. 기본 개념 실습

가장 작은 흐름은 가능 시간 조회 tool이 어떤 arguments를 받고 어떤 payload를 반환해야 하는지 확인하는 것이다.

In [ ]:
availability_payload = calendar_check_availability_payload(
    members=["민수", "지아"],
    date="2026-04-24",
)

show_json(availability_payload)

## 4. 카나메이트 확장 예제

일정 생성 tool은 조회보다 더 강한 행동이다. 최소한 요청 arguments, 생성된 `event_id`, 실행 `status`가 payload에 남아야 한다.

In [ ]:
created_event_payload = calendar_create_event_payload(
    title="발표 리허설",
    date="2026-04-24",
    start_time="15:00",
    members=["민수", "지아"],
)

show_json(created_event_payload)

## 5. 확장 예제 실행

실제 agent 실행에서는 아래처럼 tool call과 tool result가 trace에 남는다. 최종 답변보다 먼저 tool 이름과 arguments, payload를 확인한다.

In [ ]:
practice_trace = [
    {
        "event": "tool_call",
        "tool_name": "calendar_create_event",
        "arguments": created_event_payload["arguments"],
    },
    {
        "event": "tool_result",
        "tool_name": "calendar_create_event",
        "content": json.dumps(created_event_payload, ensure_ascii=False),
    },
]

assert practice_trace[0]["tool_name"] == "calendar_create_event"
assert created_event_payload["tool"] == "calendar.create_event"
assert created_event_payload["status"] == "created"

show_json(practice_trace)

## 6. 회고

확인 질문:

1. Python 함수 tool과 MCP tool은 어디에서 tool 목록을 가져온다는 점이 다른가?
2. MCP 서버를 agent 코드와 분리하면 어떤 장점이 있는가?
3. MCP payload에서 사람이 반드시 확인해야 할 필드는 무엇인가?
4. 실제 문제 repo에서 MCP tool을 agent에 연결할 때 어떤 trace를 먼저 봐야 하는가?

작은 응용 과제: 가능 시간 조회 payload와 일정 생성 payload에서 공통 필드와 생성 전용 필드를 나눠 본다.